# 🏄 momentum_swing — Qullamaggie-style breakouts and episodic pivots, daily, long only

**The idea.** Buy a stock that has already led (up `lead_min` over the `lead_window` bars *before* its base), rested in a **tightening** base of `base_window` bars, and just closed above that base on `vol_mult`× volume — or a *neglected* stock that gapped up `gap_min`+ on `gap_vol_mult`× volume and held the gap. Risk one fixed stop `stop_atr` ATRs below entry (the engine evaluates it intrabar and fills at the true open on a gap-through, so overnight gap risk is in every number), sell on the next open after the first close below the `trail_sma` SMA, and do neither unless the market — the panel's own equal-weight index — is in a rising `regime`. One shared \$10k wallet, `cash_frac` of the *remaining* free cash per entry — so there is no position cap, only geometrically shrinking clips. The parabolic short is deliberately not shipped. Full thesis, the pitfalls deep-dive, the published-evidence assessment and the \$50k account mechanics are in [README.md](./README.md).

**The universe.** `Liquid500`: 500 names ranked by median dollar volume over 2020-H1, a window that closes before the backtest opens (`notebooks/01_universe_screener.ipynb` §8b). Liquid on purpose — the modelled costs are honest there — and survivor-only (the DB holds no delisted names), so the benchmark-relative numbers are the ones to read. The train window is 2020-07 → 2024-12; 2025+ is the untouched holdout.

**What is different from the other strategy notebooks.** §4 is a **pre-registered event test** that scores every setup occurrence, market-demeaned, before any optimisation; §5 is an **ablation grid** (setup × regime × trail), not a tuning sweep; §12 audits **planned versus realised loss** per trade (the gap-risk question); §13 translates the trades to a **\$50k account** (clip sizes, dollar risk, concurrent positions, same-day round trips). Everything else is the repo flow: **load → overview → backtest + baseline → event test → inspect → ablate → optimize → boundary-check → verify → validate**.

## 1 · Load the strategy and its data

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from trading_simple.config import load_settings
from trading_simple.db import make_engine
from trading_simple import backtest as bt, jobs, optimize as opt, plots, strategies
from trading_simple.data import load_panel, symbols_with_bars

settings = load_settings()
engine = make_engine(settings)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")

strat = strategies.load("momentum_swing")
d = strat.data
syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]), d["bar"],
                         d["start"], d.get("end"), limit=d.get("max_symbols"))
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
print(f"{d['universe']}: {len(syms)} symbols, {panel.close.shape[0]:,} daily bars "
      f"{panel.close.index[0].date()} → {panel.close.index[-1].date()}  (train window)")
print("fill model:", strat.fill, "| study:", strat.optuna["study"])

## 2 · Strategy overview — hyperparameters and search ranges

| param | role in the strategy | intuition |
|---|---|---|
| `setup` | breakout / pivot / both | **the ablation knob** — which leg carries the result? |
| `lead_window`, `lead_min` | the leader move before the base (breakout) / the neglect window (pivot) | one threshold divides "leader" from "neglected" |
| `base_window`, `contraction`, `depth_max` | the base: how long, how much calmer than the move, how deep | "tight, orderly" made numeric; the literature has no directional evidence for contraction, so the search may switch it off |
| `adr_min` | average daily range floor | the setup needs movers — but high-range names are the lottery decile; searched down to 0 |
| `vol_mult`, `gap_min`, `gap_vol_mult` | volume confirmation; the gap's size and volume | the high-volume return premium; gaps must be *held* (close ≥ open) |
| `regime` (+ `regime_fast/slow/long`) | none / 10-over-20 / above-200 on the equal-weight index | the best-supported component: halves drawdowns, adds little return |
| `stop_atr`, `max_stop` | the fixed initial stop; skip setups with a wider one | bounds the per-trade risk at `cash_frac × max_stop` |
| `trail_sma`, `tp_r` | the trailing exit; an optional whole-position target in R | "sell a third at 3-5R" becomes a searchable full target (0 = trail only) |
| `cash_frac` | fraction of free cash per entry | one shared wallet — never searched, a risk decision |

The table below shows, per parameter: the **default** used by the single backtest (§3), the
hand-picked **ablation grid** (§5), and the **Optuna search range** (§7). A `—` means the stage keeps the default.

In [ ]:
def _fmt_space(spec):
    """One readable line for an optuna.space entry."""
    if spec["type"] == "categorical":
        return " | ".join(map(str, spec["choices"]))
    tail = " (log)" if spec.get("log") else (f", step {spec['step']}" if spec.get("step") else "")
    return f"{spec['low']} – {spec['high']}{tail}"

pd.DataFrame({
    "default": pd.Series(strat.params),
    "ablation grid (§5)": pd.Series({k: ", ".join(map(str, v)) for k, v in strat.sweep.items()}),
    "optuna search range (§7)": pd.Series({k: _fmt_space(s)
                                           for k, s in strat.optuna["space"].items()}),
}).fillna("—")

## 3 · One backtest at the defaults — and the baseline it has to beat

Costs are always on (commission + half-spread + slippage per leg). Signals execute at the **next bar's adjusted open**; stops are evaluated intrabar. The second figure is the **mandatory baseline comparison**: equal-weight buy-and-hold of the same 500 names over the same window, paying the same costs — `excess_cagr` and `info_ratio` are measured against it.

> **Capital model:** ONE shared \$10k wallet (engine cash-shared mode) — each entry deploys `cash_frac` of the *free* cash, so the book is never more than 100% invested. A regime-gated, selective scanner sits in cash most of the time, and against a fully invested hurdle that is a real cost, paid on purpose — *time in market is not normalised away*.

In [ ]:
pf = bt.run_strategy(strat, panel, settings=settings)
stats0 = bt.portfolio_stats(pf)
fees0, slip0 = bt.panel_costs(settings.costs, panel)     # match run_strategy's pricing
stats0.update(bt.active_stats(pf, panel, fees=fees0, slippage=slip0,
                              fill=strat.fill, total_return=stats0["total_return"]))
for k, v in stats0.items():
    print(f"  {k:16s} = {v:.4f}")
plots.plot_equity(bt.equity_curve(pf, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — equity, config defaults").show()

# The baseline: strategies/baseline (== backtest.benchmark, test-pinned) on the SAME panel.
INIT_CASH = settings.backtest.init_cash
pf_base = bt.run_strategy(strategies.load("baseline"), panel, settings=settings)
fig = go.Figure()
fig.add_trace(plots._scatter(
    bt.equity_curve(pf_base, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0,
    name="baseline (buy & hold)", line={"color": "#898781", "dash": "dash", "width": 1.5},
    hovertemplate="%{x|%Y-%m-%d}<br>%{y:+.1%}<extra>baseline</extra>"))
fig.add_trace(plots._scatter(
    bt.equity_curve(pf, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0,
    name="momentum_swing", line={"color": plots.SERIES[0], "width": 1.5},
    hovertemplate="%{x|%Y-%m-%d}<br>%{y:+.1%}<extra>momentum_swing</extra>"))
plots._base(fig, title=f"{d['universe']} · train — cumulative return vs equal-weight buy-and-hold",
            height=420)
fig.update_yaxes(tickformat="+.0%", title_text="cumulative return")
fig.show()

## 4 · The pre-registered event test — does the setup carry anything at all?

**Run this before anything else, and decide on it.** A breakout strategy's profit sits in a minority of trades, so an optimiser with thirteen knobs will fit *those* trades. The one number that does not depend on which trades is the event itself: every occurrence of each raw setup, scored by its **market-demeaned** forward return from the next open (the execution bar) over 5 / 10 / 21 / 42 bars. Subtracting each day's cross-sectional mean removes the market and leaves stock selection. The t-statistic is computed on **monthly-clustered** means, because events cluster in time and their forward windows overlap.

Three populations: the raw **breakout** and the raw **pivot** (ungated — the setup as such), and the **gated signal** the backtest actually trades. **Pass** = the gated signal's 10- and 21-day means are positive with t > 2. **Fail** = stop here; write the result into `IDEAS.md`; do not run §7. The verdict is printed, not implied.

In [ ]:
HORIZONS = (5, 10, 21, 42)
setups = strat.signals.__globals__["setups"]   # public helper on the strategy module

def event_table(mask: pd.DataFrame, close: pd.DataFrame, open_: pd.DataFrame) -> pd.DataFrame:
    """Market-demeaned forward return after each True cell of `mask`, entered at the next open.

    Demeaning each day's forward returns cross-sectionally removes the market. The t-stat
    uses monthly-clustered means: events cluster in time and overlap, so per-event t-stats
    would be inflated several-fold.
    """
    rows = {}
    for h in HORIZONS:
        fwd = close.shift(-h) / open_.shift(-1) - 1.0            # next-open entry, held h bars
        rel = fwd.sub(fwd.mean(axis=1), axis=0)                    # <- the market, removed
        ev = rel.where(mask).stack()
        by_month = ev.groupby(ev.index.get_level_values(0).to_period("M")).mean()
        t = (by_month.mean() / (by_month.std() / np.sqrt(len(by_month)))
             if len(by_month) > 2 and by_month.std() else np.nan)
        rows[f"{h}d"] = {"events": len(ev), "mean": ev.mean(), "median": ev.median(),
                         "hit_rate": (ev > 0).mean(), "t (monthly)": t, "months": len(by_month)}
    return pd.DataFrame(rows).T

FMT = {"events": "{:.0f}", "mean": "{:+.2%}", "median": "{:+.2%}", "hit_rate": "{:.0%}",
       "t (monthly)": "{:+.2f}", "months": "{:.0f}"}
parts = setups(panel, **strat.params)
adr_share = (parts["adr"] >= strat.params["adr_min"]).mean().mean()
print(f"regime '{strat.params['regime']}' is ON {parts['regime_ok'].mean():.0%} of days; "
      f"ADR ≥ {strat.params['adr_min']:.1%} on {adr_share:.0%} of name-days")
for label, key in (("BREAKOUT — raw setup, ungated", "breakout"),
                   ("EPISODIC PIVOT — raw setup, ungated", "pivot"),
                   ("GATED SIGNAL — what the backtest trades", "signal")):
    print(f"\n{label}: {int(parts[key].to_numpy().sum())} events on "
          f"{int((parts[key].sum() > 0).sum())} names")
    display(event_table(parts[key], panel.close, panel.open_).style.format(FMT))

gated = event_table(parts["signal"], panel.close, panel.open_)
ok = all(gated.loc[h, "mean"] > 0 and gated.loc[h, "t (monthly)"] > 2 for h in ("10d", "21d"))
print("\nVERDICT:", "PASS — the traded event carries a market-relative edge at 10 and 21 days"
      if ok else "FAIL — the traded event carries no significant market-relative edge; "
                 "do not run §7, write the result into IDEAS.md")

## 5 · Inspect a symbol — trades and overlays

`inspect_symbol` is the microscope of this notebook, used twice: here on the most-traded symbol with the *default* parameters, and again in §11 on the *optimized* ones. **Price panel:** the trailing SMA (the exit line) and the prior base high (the line a breakout must close above); ▲ = entry, ✕ = exit; the entry→exit segment is green when the trade won, red when it lost. **Volume panel:** amber dots mark the bars whose volume cleared `vol_mult` × the 20-day average — the confirmation both setups need.

In [ ]:
def inspect_symbol(sym, pf, params):
    """Deep-dive one symbol of a portfolio: annotated chart + its trade table."""
    all_trades = bt.trade_records(pf)
    merged = {**strat.params, **params}
    overlays = strat.indicators(panel.close[sym], panel.volume[sym], **merged)
    plots.plot_symbol(panel.close[sym], panel.volume[sym], all_trades, symbol=sym,
                      overlays=overlays, equity=pf.value(group_by=False)[sym]).show()
    mine = all_trades[all_trades["symbol"] == sym]
    return mine.sort_values("entry_ts").reset_index(drop=True).round(3)

trades = bt.trade_records(pf)
busiest = trades["symbol"].value_counts().index[0]
inspect_symbol(busiest, pf, strat.params)

## 6 · Ablation grid — which leg, which gate, which trail?

Not a tuning sweep: the grid crosses `setup` (breakout / pivot / both) with `regime` (none / fast / slow) and `trail_sma` (10 / 20), everything else at the shipped defaults — 18 combos. Read it as three questions. Does either leg carry the result alone, and do they add up? Does the market gate buy anything (it should show up in `max_drawdown`, not in `excess_cagr`)? Does the slower trail pay for the give-back it costs?

In [ ]:
grid = bt.grid_from_config(strat.sweep, strat.valid)
print(f"{len(grid)} combos")
sweep_stats = bt.sweep(strat, panel, grid, settings=settings)
cols = ["setup", "regime", "trail_sma", "excess_cagr", "cagr", "sharpe", "max_drawdown",
        "pct_profitable", "n_trades", "trades_per_year"]
sweep_stats[cols].sort_values("excess_cagr", ascending=False).round(3)

In [ ]:
plots.plot_sweep(sweep_stats).show()
plots.plot_heatmap_grid(
    sweep_stats,
    pairs=[("setup", "regime"), ("trail_sma", "regime")],
    metrics=("excess_cagr", "max_drawdown", "pct_profitable"),
).show()

## 7 · Optimize (resumable, runs outside the notebook) — only if §4 passed

`jobs.run_optimize` hands the work to a `tsim optimize` subprocess. The study scores **`excess_cagr` (maximized) against `tracking_error` (minimized)**, both relative to buy-and-hold, so the result is a *Pareto front*. Trials trading below `optuna.min_trades_per_year` (20/yr) are pruned in-objective. Every finished trial is checkpointed to `studies/{study}.log`; **re-running this cell resumes**.

> **One worker.** Several workers on one study share the config `seed` and propose the *same* parameter sequence (v1: 450 trials from 3 workers, 300 duplicates) — see `IDEAS.md` #2. Keep `workers=1` until the sampler seed is per-process.

In [ ]:
N_TRIALS = 400   # raise for a deeper search; re-runs accumulate into the same study
jobs.run_optimize("momentum_swing", n_trials=N_TRIALS)

## 8 · Read the study — admissibility before picking

Few-trade parameter sets can post a flattering number by luck, so the front is filtered to trials with at least `MIN_TRADES` before choosing (the in-objective floor guards new trials; this guards older ones in resumed journals). Among the admissible front we take the repo's one balanced pick (`opt.PICK_WEIGHTS`), which only ever chooses a row that **beats buy-and-hold**; if none does, the pick is flagged and everything downstream is a diagnosis, not a strategy.

In [ ]:
MIN_TRADES = 90     # 20 trades/yr x 4.5 train years
study = opt.load_study(strat)
front = opt.pareto_frame(study)
everything = opt.trials_frame(study)
admissible = front[front["n_trades"] >= MIN_TRADES].copy()
print(f"trials: {len(study.trials)} total, {len(everything)} scored; front: {len(front)}, "
      f"admissible (n_trades ≥ {MIN_TRADES}): {len(admissible)}")

_INTS = ("lead_window", "base_window", "trail_sma")
def coerce(params):
    """Cast params back to real types (front rows store everything as object)."""
    return {k: (int(round(float(v))) if k in _INTS else v if isinstance(v, str) else float(v))
            for k, v in params.items()}

X, Y = opt.OBJECTIVES                       # (excess_cagr, tracking_error)
pool = admissible if len(admissible) else front
best_row = opt.pick_from_front(pool)
print(f"{int(best_row['front_distinct'])} distinct outcomes on the admissible front"
      + (f" — picked one of {int(best_row['n_tied'])} TIED trials, arbitrarily"
         if best_row["n_tied"] > 1 else ""))
picked = coerce({k: best_row[k] for k in strat.optuna["space"]})
beats = front["excess_cagr"] > opt.MIN_PICK_EXCESS
print(f"trials on the front that BEAT buy-and-hold: {int(beats.sum())} of {len(front)}"
      + ("" if best_row["beats_benchmark"] else " — none do, so the pick is flagged"))
print("picked params:", picked)
print("picked train metrics:", {m: round(float(best_row[m]), 4)
      for m in (X, Y, "info_ratio", "sharpe", "cagr", "max_drawdown", "pct_profitable",
                "n_trades", "trades_per_year", "years")})
plots.plot_pareto(front, everything, picked=int(best_row["trial"])).show()
print("\nmean excess_cagr of scored trials by setup × regime (does either leg or gate help?):")
by_leg = everything.groupby(["setup", "regime"])["excess_cagr"]
display(by_leg.agg(["mean", "max", "count"]).round(3))

## 9 · Boundary check — did the optimum hit the search-space edge?

A picked trial sitting on the edge of a searched range hints that the optimum may lie outside it. `optimize.boundary_report` compares the picked parameters (and the whole front) with each parameter's `search:` range — log-aware — and flags anything within 5% of a bound. If something is flagged: widen that bound in `config.yaml`, **bump `optuna.study`**, re-run §7, and re-read §8–§9 until the front sits inside the box.

In [ ]:
report = opt.boundary_report(picked, strat.optuna["space"], front=front)
flagged = report[report["action"] != "ok"]
print("all clear — the searched ranges bracket the optimum" if flagged.empty
      else f"{len(flagged)} parameter(s) pressing against the search space:")
report

## 10 · Backtest the picked parameters

In [ ]:
pf_best = bt.run_strategy(strat, panel, params=picked, settings=settings)
stats_best = bt.portfolio_stats(pf_best)
stats_best.update(bt.active_stats(pf_best, panel, fees=fees0, slippage=slip0,
                                  fill=strat.fill, total_return=stats_best["total_return"]))
display(pd.Series(stats_best).round(4).to_frame("picked (train)"))
plots.plot_equity(bt.equity_curve(pf_best, init_cash=INIT_CASH),
                  title=f"{strat.name} — equity with picked parameters").show()

## 11 · Verification — per-symbol deep dive

Portfolio means can hide an edge carried by a single name, so verify the picked parameters symbol by symbol. In wallet mode a symbol's `total_return` is its **contribution** to the one book, not a standalone return. Then set `SYMBOL` to any name that looks interesting (the big winner, the worst loser) and re-run the second cell for the annotated chart and trade table.

In [ ]:
bt.symbol_stats(pf_best).round(4).head(15)

In [ ]:
SYMBOL = None   # ← set to any symbol from the table above; None = most-traded
sym = SYMBOL or bt.trade_records(pf_best)["symbol"].value_counts().index[0]
inspect_symbol(sym, pf_best, picked)

## 12 · Validate — is the edge real?

Everything above is in-sample. **§12a can kill the strategy outright**: a long-only book in a rising market posts a positive absolute Sharpe by being invested, so the only question that counts is whether it beats simply owning the universe. The holdout and the per-year table are read off a **single run over the full stored history**, sliced *after* the fact — a strategy with a 200-bar regime SMA and a 63-bar leader window would otherwise spend the first ten months of a sliced holdout in warm-up, which is not a test of anything.

In [ ]:
full = load_panel(engine, syms, d["bar"], d["start"], None,
                  session=d.get("session", "regular"), exchanges=settings.exchanges)
print(f"full history: {full.close.shape[0]:,} bars "
      f"{full.close.index[0].date()} → {full.close.index[-1].date()}")
HOLDOUT_START = "2025-01-01"
_full_runs = {}

def window_stats(params, start, end, label=None):
    """Strategy vs the equal-weight hurdle inside [start, end], from ONE run over the full history.

    The equity curve and the benchmark's per-bar returns are computed once over the whole
    stored history and sliced, so the window carries no indicator warm-up and positions
    open at its start are held exactly as a live account would hold them.
    """
    key = tuple(sorted(params.items()))
    if key not in _full_runs:
        pf_f = bt.run_strategy(strat, full, params=params, settings=settings)
        fees, slip = bt.panel_costs(settings.costs, full)
        _, bench = bt.benchmark(full, fees=fees, slippage=slip, fill=strat.fill)
        _full_runs[key] = (bt.equity_curve(pf_f)["value"], bench, bt.trade_records(pf_f))
    value, bench, tr = _full_runs[key]
    m = ((value.index >= pd.Timestamp(start, tz="UTC"))
         & (value.index <= pd.Timestamp(end, tz="UTC")))
    v, b = value[m], bench[m]
    years = bt._years(v.index)
    r = v.pct_change().fillna(0.0)
    active = r - b
    ann = np.sqrt(len(v) / years)
    st = {
        "cagr": bt._cagr(v.iloc[-1] / v.iloc[0] - 1.0, years),
        "bench_cagr": bt._cagr(float((1.0 + b).prod()) - 1.0, years),
        "info_ratio": float(active.mean() / active.std() * ann) if active.std() else 0.0,
        "tracking_error": float(active.std() * ann),
        "max_drawdown": float((v / v.cummax() - 1.0).min()),
        "n_trades": int(((tr["entry_ts"] >= v.index[0]) & (tr["entry_ts"] <= v.index[-1])).sum()),
        "years": years,
    }
    st["excess_cagr"] = st["cagr"] - st["bench_cagr"]
    if label:
        ok = st["excess_cagr"] > 0 and st["info_ratio"] > 0
        print(f"\n{label}  ({v.index[0].date()} -> {v.index[-1].date()}, {years:.2f} years, "
              f"{st['n_trades']} trades entered)")
        print(f"  strategy     {st['cagr']:+7.2%}/yr   max drawdown {st['max_drawdown']:.1%}")
        print(f"  buy & hold   {st['bench_cagr']:+7.2%}/yr")
        print(f"  EXCESS       {st['excess_cagr']:+7.2%}/yr   "
              f"information ratio {st['info_ratio']:+5.2f}")
        print("  -->", "PASS — it beats owning the universe" if ok
              else "FAIL — owning the universe was better")
    return st

### 12a · The hurdle: does it beat simply owning the universe?

Train first (the in-sample number the optimiser saw, via `bt.evaluate` on the train panel), then the **untouched 2025+ holdout** from the full-history run. Both must pass before the pick means anything.

In [ ]:
st_train = bt.evaluate(strat, panel, picked, settings=settings)
fees_t, slip_t = bt.panel_costs(settings.costs, panel)
bench_t, _ = bt.benchmark(panel, fees=fees_t, slippage=slip_t, fill=strat.fill)
print(f"TRAIN  ({panel.close.index[0].date()} -> {panel.close.index[-1].date()}, "
      f"{st_train['years']:.2f} years, {st_train['trades_per_year']:.0f} trades/yr)")
print(f"  strategy     {st_train['cagr']:+7.2%}/yr   sharpe {st_train['sharpe']:5.2f}   "
      f"max drawdown {st_train['max_drawdown']:.1%}   n_trades {int(st_train['n_trades'])}")
print(f"  buy & hold   {bt._cagr(bench_t, st_train['years']):+7.2%}/yr")
print(f"  EXCESS       {st_train['excess_cagr']:+7.2%}/yr   "
      f"information ratio {st_train['info_ratio']:+5.2f}")
ok_train = st_train["excess_cagr"] > 0 and st_train["info_ratio"] > 0
print("  -->", "PASS — it beats owning the universe" if ok_train
      else "FAIL — owning the universe was better")
st_hold = window_stats(picked, HOLDOUT_START, str(full.close.index[-1].date()), "HOLDOUT (2025+)")

### 12b · Costs on, then doubled — the breakeven, and the two fixed bars

Re-run the pick at 1×/2×/3× the configured costs, find the round-trip cost at which the Sharpe hits zero, then score it at the repo's two fixed bars — **6/5 bps** and **10/3 bps** (spread/slippage per leg) — with the benchmark re-priced at the same costs. Liquid500 at 9 bps round trip is the friendliest case this style will ever see; the source's small-cap universe is 30–100 bps.

In [ ]:
def with_costs(**overrides):
    """A copy of settings with its (pydantic) cost block overridden."""
    return settings.model_copy(update={"costs": settings.costs.model_copy(update=overrides)})

c = settings.costs
for mult in (1.0, 2.0, 3.0):
    s = with_costs(spread_bps=c.spread_bps * mult, slippage_bps=c.slippage_bps * mult,
                   commission_per_share=c.commission_per_share * mult)
    st = bt.portfolio_stats(bt.run_strategy(strat, panel, params=picked, settings=s))
    print(f"  {mult:.0f}x costs: sharpe={st['sharpe']:.3f}  return={st['total_return']:.2%}")

base_bps = c.spread_bps + c.slippage_bps
breakeven = None
for extra in np.arange(0, 60, 2.0):
    s = with_costs(spread_bps=base_bps + extra, slippage_bps=0.0)
    st = bt.portfolio_stats(bt.run_strategy(strat, panel, params=picked, settings=s))
    if st["sharpe"] <= 0:
        breakeven = base_bps + extra
        break
be = ">60" if breakeven is None else round(breakeven, 1)
print(f"\napprox breakeven half-spread: {be} bps per leg")

print("\nfixed cost bars (spread/slippage bps per leg), benchmark re-priced at the same costs:")
for spread, slip in ((6.0, 5.0), (10.0, 3.0)):
    s = with_costs(spread_bps=spread, slippage_bps=slip)
    st = bt.evaluate(strat, panel, picked, settings=s)
    print(f"  {spread:>4.0f}/{slip:<3.0f} bps: excess_cagr {st['excess_cagr']:+.2%}/yr   "
          f"info_ratio {st['info_ratio']:+.2f}   n_trades {int(st['n_trades'])}")

### 12c · Sub-period consistency

One number over four years can hide a strategy that only worked in one regime. Each calendar year from the full-history run (no warm-up holes); **2022 is the year to read** — the bear year inside the window, and the year the source's own account gave back most of its gains.

In [ ]:
years = []
last = full.close.index[-1]
for y in range(2020, last.year + 1):
    start = f"{y}-07-01" if y == 2020 else f"{y}-01-01"
    end = f"{y}-12-31" if y < last.year else str(last.date())
    st = window_stats(picked, start, end)
    keep = ("cagr", "bench_cagr", "excess_cagr", "info_ratio", "max_drawdown", "n_trades")
    years.append({"year": y, **{k: round(float(st[k]), 3) for k in keep}})
display(pd.DataFrame(years).set_index("year"))

### 12d · Untouched holdout (open once)

2025+ was never seen by the config, the grid, or the optimizer. The numbers are already in §12a; this is the picture. If it disappoints, that is information — do not tune against it.

In [ ]:
key = tuple(sorted(picked.items()))
value, bench, _ = _full_runs[key]
m = value.index >= pd.Timestamp(HOLDOUT_START, tz="UTC")
fig = go.Figure()
fig.add_trace(plots._scatter((1.0 + bench[m]).cumprod() - 1.0, name="baseline (buy & hold)",
                             line={"color": "#898781", "dash": "dash", "width": 1.5},
                             hovertemplate="%{x|%Y-%m-%d}<br>%{y:+.1%}<extra>baseline</extra>"))
fig.add_trace(plots._scatter(value[m] / value[m].iloc[0] - 1.0, name=strat.name,
                             line={"color": plots.SERIES[0], "width": 1.5},
                             hovertemplate=("%{x|%Y-%m-%d}<br>%{y:+.1%}<extra>"
                                            + strat.name + "</extra>")))
plots._base(fig, title="HOLDOUT 2025+ — cumulative return vs equal-weight buy-and-hold", height=420)
fig.update_yaxes(tickformat="+.0%", title_text="cumulative return")
fig.show()

## 13 · Gap-risk audit — planned versus realised loss

The brief's central caution: the stop is a day-session stop, the position is held overnight, and a stock can gap through it. The engine models exactly that (the stop leg fills at the true open when a bar opens below it), so the realised loss of every trade can be compared with the loss that was *planned* at entry (`stop_atr` × ATR). Read: the share of losing trades that lost more than planned, by how much, and the worst one. Under fixed-fraction sizing the planned book risk is `cash_frac` × the stop; a 20% gap through a 12% stop costs 5% of the book.

In [ ]:
sig_best = strat.signals(panel, **{**strat.params, **picked})
tr = bt.trade_records(pf_best).copy()
tr["planned_stop"] = [float(sig_best["sl_stop"].loc[ts, s])
                      for ts, s in zip(tr["entry_ts"], tr["symbol"], strict=True)]
tr["hold_days"] = (tr["exit_ts"] - tr["entry_ts"]).dt.days
losers = tr[tr["return"] < 0].copy()
losers["beyond_plan"] = -losers["return"] - losers["planned_stop"]
worse = losers[losers["beyond_plan"] > 0.001]
print(f"{len(tr)} trades, {len(losers)} losers ({len(losers) / max(len(tr), 1):.0%}), "
      f"median hold {tr['hold_days'].median():.0f} days, "
      f"median planned stop {tr['planned_stop'].median():.1%}")
print(f"losers that lost MORE than planned: {len(worse)} of {len(losers)} "
      f"({len(worse) / max(len(losers), 1):.0%}); "
      f"median overshoot {worse['beyond_plan'].median():+.1%}, "
      f"worst {worse['beyond_plan'].max():+.1%} beyond a "
      f"{worse.loc[worse['beyond_plan'].idxmax(), 'planned_stop']:.1%} stop"
      if len(worse) else "no loser exceeded its planned stop")
print(f"book impact: planned risk per trade = cash_frac × stop = "
      f"{strat.params['cash_frac'] * tr['planned_stop'].median():.2%} of the book (median); "
      f"worst realised single-trade loss = {tr['return'].min():+.1%} of the position")
print("\nreturn distribution per trade (fraction of the position):")
display(tr["return"].describe(percentiles=[.05, .1, .25, .5, .75, .9, .95])
        .round(4).to_frame("return"))
print("\nworst 8 trades:")
worst_cols = ["symbol", "entry_ts", "exit_ts", "hold_days", "planned_stop", "return", "pnl"]
display(tr.sort_values("return")[worst_cols].head(8).round(3))

## 14 · The same trades at \$50,000

The \$10k book is a reporting scale (fractional shares, one shared wallet), so every number scales linearly. This cell translates the picked backtest to a \$50k account: clip sizes (each entry takes 25% of the *remaining* cash: 25%, 19%, 14%, 11%, 8%… — there is no position cap, so read the concurrent-positions line), planned dollar risk, how many positions were held at once, and the **same-day round trips** (bought at the open, stopped the same day — the only trades that would ever count as day trades). The historic PDT rule is not binding at \$50k and is being replaced (FINRA Regulatory Notice 26-10, effective 4 June 2026, broker phase-in to 20 Oct 2027); the count is here so a sub-\$25k reader can see whether it would bind for them.

In [ ]:
ACCOUNT = 50_000.0
scale = ACCOUNT / INIT_CASH
tr["position_$"] = tr["size"] * tr["entry_price"] * scale
tr["planned_risk_$"] = tr["position_$"] * tr["planned_stop"]
tr["pnl_$"] = tr["pnl"] * scale
held = pd.DataFrame(False, index=panel.close.index, columns=sorted(tr["symbol"].unique()))
for _, t in tr.iterrows():
    held.loc[t["entry_ts"]:t["exit_ts"], t["symbol"]] = True
concurrent = held.sum(axis=1)
same_day = tr[tr["entry_ts"] == tr["exit_ts"]]
per_day = same_day.groupby("entry_ts").size().reindex(panel.close.index, fill_value=0)
print(f"account ${ACCOUNT:,.0f}: {len(tr)} trades over {st_train['years']:.1f} years "
      f"({st_train['trades_per_year']:.0f}/yr)")
print(f"  position size   median ${tr['position_$'].median():,.0f}, "
      f"max ${tr['position_$'].max():,.0f}")
print(f"  planned risk    median ${tr['planned_risk_$'].median():,.0f} "
      f"({tr['planned_risk_$'].median() / ACCOUNT:.2%} of the account), "
      f"max ${tr['planned_risk_$'].max():,.0f}")
print(f"  realised P&L    total ${tr['pnl_$'].sum():+,.0f}; "
      f"best ${tr['pnl_$'].max():+,.0f}, worst ${tr['pnl_$'].min():+,.0f}")
print(f"  concurrent positions: max {int(concurrent.max())}, "
      f"mean on days with any {concurrent[concurrent > 0].mean():.1f}, "
      f"in the market {(concurrent > 0).mean():.0%} of days")
print(f"  same-day round trips: {len(same_day)} in total; max in any 5-session window: "
      f"{int(per_day.rolling(5).sum().max())}  (the old PDT limit was 4)")

## Read the result honestly

* **§4 decides whether there is anything to trade**; §12a decides whether the *strategy* harvests it after costs against owning the universe. Both must hold, on train, before the holdout is opened once.
* **The base rate is against this.** The one faithful public rebuild of these rules earns 1.7%/yr against SPY's 13.5%; the rigorous trend-following-on-stocks results (Wilcox & Crittenden; Zarattini, Pagani & Wilcox) are real but carried by under 7% of trades. A documented negative here is a legitimate outcome — see `turnover_momentum` and `gap_and_go`.
* **Time in market is a real cost.** A regime-gated scanner that is 40% invested must earn back the 60% it sat out against a fully invested hurdle; the objectives price that correctly and on purpose. Read the gate's value in `max_drawdown` and the 2022 row of §12c, not in `excess_cagr`.
* **Judge breadth on `pct_profitable`**, read `n_trades` beside every Sharpe, and treat any front row that only exists at a corner of the search space (§9) as noise.
* **The universe is survivor-only** — failed breakouts in names that later delisted are the losses that are missing. Absolute returns and drawdowns are flattered; the benchmark-relative numbers are the honest ones.
* **What the engine cannot model** (README §2i): risk-per-trade sizing, partial sells, opening-range entries, earnings versus non-earnings gaps, borrow costs. The first is an engine proposal in `IDEAS.md`; the last is why the parabolic short is not here.
* **Expected magnitude, honestly:** the rigorous long-breakout results are +5–6%/yr of alpha *gross* in liquid US stocks with 30–45% drawdowns. The source's numbers are a different population, execution and instrument.